# Atelier Pratique : De la régression à l'IA Explicable (XAI)

**Durée estimée :** 15 à 20 minutes | **Niveau :** Initiation (aucun prérequis en programmation)

### Objectifs pédagogiques
1. Comprendre pourquoi une régression linéaire sur de vraies données (`ANES 2024`, `N = 4 734`) est un modèle **« boîte de verre »** dont chaque paramètre s'interprète directement en dollars par an (`$ / an`).
2. Auditer des **biais algorithmiques** sur deux attributs sensibles (`female` et `minority`), manipuler la logique **« toutes choses égales par ailleurs »** et repérer les **variables proxy** grâce aux contributions locales (principe fondateur de l'XAI).
3. Situer le passage des modèles explicables (`2` à `7` paramètres) aux réseaux de neurones boîtes noires (`329` paramètres) et aux grands modèles de langage (`70 Md+` de paramètres).

> **Première fois sur Google Colab ?**
> - **Exécuter une cellule :** placez votre souris sur une cellule de code et cliquez sur le bouton **▶** à gauche (ou appuyez sur `Maj + Entrée`).
> - **Rien à installer :** `numpy`, `pandas`, `statsmodels` et `matplotlib` sont déjà préinstallés dans Google Colab.
> - **Enregistrer votre travail :** cliquez sur *Fichier > Enregistrer une copie dans Drive* si vous voulez conserver vos propres essais.
> - **Ordre d'exécution :** lancez simplement les cellules une par une, de haut en bas.

## Étape 1 : Charger les vraies données `ANES 2024` (`N = 4 734`)

Pour cet atelier, nous travaillons sur un jeu de données **réel et contemporain** issu de la grande enquête américaine *American National Election Studies 2024* (`ANES 2024`, `N = 4 734` répondant·es adultes aux États-Unis), chargé directement depuis le dépôt public `datamisc/ts-2024`.

Dans l'enquête d'origine, le revenu annuel du foyer est relevé en **28 tranches officielles** (`V241566x`), que nous convertissons aux centres officiels des tranches, de `2 500 $ / an` (moins de `5 000 $`) à `275 000 $ / an` (`250 000 $` et plus).

Voici les `6` variables explicatives et la variable cible préparées pour l'atelier :

- `education` : années d'études équivalentes (`6` à `20` ans ; `12` = Bac / *High school*, `16` = Licence / *Bachelor*, `18` = Master, `20` = Doctorat)
- `age` : âge du ou de la répondant·e (`18` à `80` ans)
- `employed` : statut d'activité (`0` = hors emploi, retraite ou études, `1` = en emploi)
- `married` : situation maritale (`0` = non marié·e, `1` = marié·e)
- `female` *(audit de biais)* : sexe (`0` = homme, `1` = femme, `53.2 %` de l'échantillon)
- `minority` *(audit de biais)* : origine / race (`0` = blanc non hispanique, `1` = minorité racisée, `27.5 %` de l'échantillon)
- `annual_income` *(variable cible)* : revenu annuel réel du foyer en dollars US de 2024 (`$ / an`)

La cellule ci-dessous charge les `7` colonnes utiles du fichier compressé et range ces `4 734` profils réels dans un `DataFrame` (un tableau de données type Excel en Python) avec la même préparation exacte que les diapositives et l'application interactive.

In [ ]:
#@title Charger le jeu de données réel ANES 2024 (N = 4 734) - cliquez sur ▶
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
RAW_COLS = ["V241566x", "V241463", "V241458x", "V241488x", "V241461x", "V241550", "V241501x"]
raw_df = pd.read_csv(DATA_URL, compression="gzip", usecols=RAW_COLS, low_memory=False)

# 28 tranches officielles ANES 2024 (V241566x) converties aux centres en milliers de $ / an
MIDPOINTS_K = {
    1: 2.5, 2: 7.5,
    3: 11.25, 4: 13.75, 5: 16.25, 6: 18.75, 7: 21.25, 8: 23.75, 9: 26.25, 10: 28.75,
    11: 32.5, 12: 37.5, 13: 42.5, 14: 47.5, 15: 52.5, 16: 57.5,
    17: 62.5, 18: 67.5, 19: 72.5, 20: 77.5, 21: 85.0, 22: 95.0,
    23: 105.0, 24: 117.5, 25: 137.5, 26: 162.5, 27: 212.5, 28: 275.0,
}

# Recodage du diplôme ANES (V241463) en années d'études équivalentes
EDUCATION_YEARS = {
    1: 6, 2: 8, 3: 9, 4: 10, 5: 11, 6: 11, 7: 11, 8: 11,
    9: 12, 10: 13, 11: 14, 12: 14, 13: 16, 14: 18, 15: 19, 16: 20,
}


def clean_anes2024(input_df: pd.DataFrame) -> pd.DataFrame:
    """Filter valid respondents and build clean English variables + annual_income."""
    subset_df = input_df[
        (input_df["V241566x"].isin(MIDPOINTS_K))
        & (input_df["V241458x"] >= 18)
        & (input_df["V241463"].isin(EDUCATION_YEARS))
        & (input_df["V241488x"] > 0)
        & (input_df["V241461x"] > 0)
        & (input_df["V241550"].isin([1, 2]))
        & (input_df["V241501x"] > 0)
    ].copy()

    return pd.DataFrame(
        {
            "education": subset_df["V241463"].map(EDUCATION_YEARS).astype(int),
            "age": subset_df["V241458x"].astype(int),
            "employed": (subset_df["V241488x"] == 1).astype(int),
            "married": (subset_df["V241461x"] == 1).astype(int),
            "female": (subset_df["V241550"] == 2).astype(int),
            "minority": (subset_df["V241501x"] != 1).astype(int),
            "annual_income": (subset_df["V241566x"].map(MIDPOINTS_K) * 1000.0).round(0),
        }
    ).reset_index(drop=True)


df = clean_anes2024(raw_df)

mean_income = float(df["annual_income"].mean())
median_income = float(df["annual_income"].median())
std_income = float(df["annual_income"].std())
max_income = float(df["annual_income"].max())

print(f"Jeu de données réel ANES 2024 chargé : {df.shape[0]} répondant·es et {df.shape[1]} colonnes.")
print(
    f"Revenu annuel moyen : {mean_income:,.0f} $ / an | "
    f"Médiane : {median_income:,.0f} $ / an | "
    f"Écart-type : {std_income:,.0f} $ / an | "
    f"Tranche haute : {max_income:,.0f} $ / an"
    .replace(",", " ")
)
print(
    f"Effectifs observés : {int(df['female'].sum())} femmes ({df['female'].mean() * 100:.1f} %), "
    f"{int(df['minority'].sum())} personnes issues d'une minorité racisée ({df['minority'].mean() * 100:.1f} %)."
)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

mean_income = float(df["annual_income"].mean())
median_income = float(df["annual_income"].median())
max_income = float(df["annual_income"].max())

fig, ax = plt.subplots(figsize=(9.0, 4.4))
ax.hist(df["annual_income"], bins=24, color="#0072bc", edgecolor="white", alpha=0.88)
ax.axvline(
    mean_income,
    color="#dd1019",
    linestyle="dashed",
    linewidth=2.2,
    label=f"Moyenne = {mean_income:,.0f} $ / an".replace(",", " "),
)
ax.axvline(
    median_income,
    color="#1e8449",
    linestyle="dotted",
    linewidth=2.4,
    label=f"Médiane = {median_income:,.0f} $ / an".replace(",", " "),
)

ax.annotate(
    f"Tranche haute : {max_income:,.0f} $ / an\n(250 000 $ et plus)".replace(",", " "),
    xy=(270000, 366),
    xytext=(182000, 435),
    arrowprops=dict(arrowstyle="->", color="#333333", lw=1.4),
    fontsize=9.5,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#333333", alpha=0.92),
)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{int(x):,} $".replace(",", " ")))
ax.set_title("Distribution du revenu annuel réel dans ANES 2024 (N = 4 734, USA 2024)", fontsize=13, fontweight="bold")
ax.set_xlabel("Revenu annuel du foyer (annual_income, en $ US / an)")
ax.set_ylabel("Nombre de répondant·es")
ax.set_ylim(0, 640)
ax.legend(loc="upper center")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Étape 2 : Un modèle « boîte de verre » à 1 seul levier (`annual_income ~ education`)

En Python avec `statsmodels`, la formule `"annual_income ~ education"` ajuste une **régression linéaire OLS** (*Ordinary Least Squares*, méthode des moindres carrés : on trace la droite la plus proche possible de tous les points) et se lit simplement : **« expliquer le revenu annuel par le nombre d'années d'études »**.

- À gauche du symbole `~` se trouve la variable que l'on veut prédire (`annual_income`).
- À droite du symbole `~` se trouvent les variables explicatives (ici, uniquement `education`).

Ce modèle ne contient que **2 paramètres** (une constante de départ et une pente) : on peut le comprendre intégralement sans aucun outil complexe.

In [ ]:
model_simple = smf.ols("annual_income ~ education", data=df).fit()

const_simple = float(model_simple.params["Intercept"])
slope_simple = float(model_simple.params["education"])
r2_simple = float(model_simple.rsquared)
rmse_simple = float(np.sqrt(np.mean((df["annual_income"] - model_simple.fittedvalues) ** 2)))
pval_simple = float(model_simple.pvalues["education"])

print(f"Constante de départ (Intercept) = {const_simple:,.1f} $ / an (valeur théorique si education = 0)".replace(",", " "))
print(f"Pente education = {slope_simple:+,.1f} $ / an par année d'études".replace(",", " "))
print(f"R² (part des écarts expliquée par le modèle, de 0 à 1) = {r2_simple:.2f} ({r2_simple * 100:.1f} %)")
print(f"RMSE (erreur moyenne typique des prédictions) = {rmse_simple:,.0f} $ / an".replace(",", " "))
print()
gain_4y_str = f"{round(slope_simple, 1) * 4:+,.0f}".replace(",", " ")
print(
    f"Interprétation : en moyenne, +4 années d'études supplémentaires "
    f"sont associées à un revenu annuel supérieur de {gain_4y_str} $ / an."
)
print(
    f"(Glossaire : Erreur std = marge d'incertitude sur le coefficient ; "
    f"p-valeur < 0.05 indique un lien statistiquement solide, ici {pval_simple:.2e} = quasi zéro.)"
)

table_simple = pd.DataFrame(
    {
        "Paramètre": ["Constante de départ (Intercept)", "Années d'études (education)"],
        "Effet": [round(const_simple, 1), round(slope_simple, 1)],
        "Unité": ["$ / an", "$ / an / année"],
        "Erreur std": [round(float(model_simple.bse["Intercept"]), 1), round(float(model_simple.bse["education"]), 1)],
        "p-valeur": [f"{float(model_simple.pvalues['Intercept']):.2e}", f"{pval_simple:.2e}"],
    }
)

# Optionnel (pour les statisticien·nes curieux·ses) : décommenter la ligne ci-dessous
# print(model_simple.summary())
table_simple

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(9.0, 4.9))
ax.scatter(
    df["education"],
    df["annual_income"],
    alpha=0.18,
    s=22,
    color="#0072bc",
    label="Répondant·es réel·les ANES 2024 (N = 4 734)",
)

x_grid = np.linspace(df["education"].min(), df["education"].max(), 100)
y_line = const_simple + slope_simple * x_grid
ax.plot(
    x_grid,
    y_line,
    color="#dd1019",
    linewidth=2.5,
    label=f"Droite OLS : {const_simple:,.0f} + {slope_simple:,.0f} × education".replace(",", " "),
)

# Annotation visuelle de la pente (+4 ans -> +53 780 $ / an)
x0, x1 = 12.0, 16.0
y0 = const_simple + slope_simple * x0
y1 = const_simple + slope_simple * x1
gain_4y_str = f"{round(slope_simple, 1) * 4:+,.0f}".replace(",", " ")
ax.plot([x0, x1, x1], [y0, y0, y1], color="#1e8449", linewidth=2.2, linestyle="dashed")
ax.annotate(
    f"+4 ans -> {gain_4y_str} $ / an",
    xy=(x1, (y0 + y1) / 2),
    xytext=(7.2, 165000),
    arrowprops=dict(arrowstyle="->", color="#1e8449", lw=1.6),
    fontsize=10.5,
    fontweight="bold",
    color="#1e8449",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#1e8449", alpha=0.94),
)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, pos: f"{int(y):,} $".replace(",", " ")))
ax.set_title("Régression simple : revenu annuel selon les années d'études (ANES 2024)", fontsize=13, fontweight="bold")
ax.set_xlabel("Années d'études équivalentes (education)")
ax.set_ylabel("Revenu annuel du foyer (annual_income, $ / an)")
ax.set_ylim(-5000, 290000)
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(
    "Astuce atelier : dans l'Onglet 2 (« Une droite ») de l'application web (qui affiche 100 points réels cliquables "
    "et les résidus sur l'ensemble des points), cliquez sur le graphique pour ajouter des profils extrêmes et voir comment la droite bascule !"
)

## Étape 3 : Ajouter des leviers (`7` paramètres, toujours 100 % lisible)

Les années d'études seules n'expliquent que `17.8 %` des écarts de revenu annuel (`R² = 0.18`). Enrichissons le modèle avec les `5` autres variables réelles de `ANES 2024` : `age`, `employed`, `married`, `female` et `minority`.

Le modèle passe à **7 paramètres** (`1` constante + `6` pentes). Chaque coefficient mesure désormais l'effet propre d'une variable **toutes choses égales par ailleurs** (c'est-à-dire à niveau identique sur toutes les autres variables).

In [ ]:
model_full = smf.ols(
    "annual_income ~ education + age + employed + married + female + minority",
    data=df,
).fit()

feature_labels = {
    "Intercept": "Constante (base)",
    "education": "Années d'études (education)",
    "age": "Âge (age)",
    "employed": "En emploi (0=Hors emploi, 1=En emploi)",
    "married": "Marié·e (0=Non, 1=Oui)",
    "female": "Sexe (female : 0=Homme, 1=Femme)",
    "minority": "Origine (minority : 0=Blanc non-hisp., 1=Minorité)",
}

feature_units = {
    "Intercept": "$ / an",
    "education": "$ / an / année",
    "age": "$ / an / année",
    "employed": "$ / an",
    "married": "$ / an",
    "female": "$ / an",
    "minority": "$ / an",
}


def describe_role(key: str, coef: float) -> str:
    """Return a concise French interpretation for each linear model term."""
    if key == "Intercept":
        return "Point de départ théorique (toutes les variables à 0)"
    direction = "Augmente le revenu prédit" if coef > 0 else "Réduit le revenu prédit"
    contexts = {
        "education": "par année d'études",
        "age": "par année d'âge (tendance linéaire moyenne)",
        "employed": "si personne en emploi",
        "married": "si foyer marié (souvent deux revenus)",
        "female": "si femme (écart historique réel, p < 0.001)",
        "minority": "si minorité racisée (27.5 % de l'échantillon, p < 0.001)",
    }
    coef_str = f"{coef:+,.1f}".replace(",", " ")
    return f"{direction} ({coef_str} {feature_units[key]} {contexts[key]})"


table_rows = []
for key in model_full.params.index:
    coef = float(model_full.params[key])
    table_rows.append(
        {
            "Variable": feature_labels[key],
            "Effet": round(coef, 1),
            "Unité": feature_units[key],
            "Erreur std": round(float(model_full.bse[key]), 1),
            "p-valeur": f"{float(model_full.pvalues[key]):.2e}",
            "Rôle dans le modèle": describe_role(key, coef),
        }
    )

table_full = pd.DataFrame(table_rows)
rmse_full = float(np.sqrt(np.mean((df["annual_income"] - model_full.fittedvalues) ** 2)))

print(f"R² du modèle complet (7 paramètres) = {model_full.rsquared:.2f} ({model_full.rsquared * 100:.1f} % des écarts expliqués)")
print(f"Erreur moyenne typique (RMSE) = {rmse_full:,.0f} $ / an".replace(",", " "))
# Optionnel (pour les statisticien·nes curieux·ses) : décommenter la ligne ci-dessous
# print(model_full.summary())
table_full

### Audit de biais sur données réelles ANES 2024 : que révèlent `female` et `minority` ?

Toutes choses égales par ailleurs (à niveau d'études, d'âge, de statut d'emploi et de situation maritale identique), le modèle mesure deux écarts massifs et hautement significatifs dans `ANES 2024` (tous deux à `p < 0.001`) :

- **`female` = `-8 007 $ / an`** (`-8 007.4 $ / an`, `p = 2.24e-05 < 0.001`) : un écart de revenu réel et statistiquement robuste entre femmes et hommes à profil comparable.
- **`minority` = `-10 328 $ / an`** (`-10 328.4 $ / an`, `p = 1.93e-06 < 0.001`) : un écart pénalisant majeur sur les `1 301` répondant·es issu·es d'une minorité racisée (`27.5 %` des `4 734` profils).

**Pourquoi c'est critique en scoring et sous l'AI Act européen :**
- Dans un système réel d'aide à la décision (recrutement, octroi de crédit, tarification), un modèle entraîné sur des données historiques **reproduit mécaniquement les inégalités constatées dans la société**. L'AI Act européen classe ces usages à haut risque et impose audit de biais, traçabilité et gouvernance des données.
- **Le piège des variables proxy (comme dans le cas COMPAS) :** supprimer simplement les colonnes `female` et `minority` ne suffit pas à rendre une boîte noire neutre ! Dans `ANES 2024`, d'autres variables restent corrélées à ces attributs : par exemple `married` ne concerne que `40.1 %` des répondant·es `minority = 1` contre `56.0 %` des répondant·es `minority = 0` (et `47.2 %` des femmes contre `56.7 %` des hommes), `education` vaut en moyenne `14.2` ans contre `14.9` ans selon `minority`, et `employed` diffère également (`66.7 %` pour les femmes contre `71.1 %` pour les hommes). Un réseau de neurones peut donc reconstruire partiellement l'attribut supprimé via ces **variables proxy**.

## Étape 4 : Hack-Time : Créez un profil et auditez la prédiction !

Prenez la main sur le modèle : ajustez les curseurs du formulaire ci-dessous pour inventer un profil, puis cliquez sur **▶** pour exécuter la cellule.

Le code calcule le revenu annuel prédit (borné entre `2 500 $ / an` et `275 000 $ / an`) et le décompose **variable par variable** par rapport à la moyenne de l'échantillon (`97 347 $ / an`), selon la formule exacte :

`contribution(variable) = coefficient × (valeur_choisie - moyenne_échantillon)`

C'est exactement l'idée au cœur de nombreuses méthodes d'IA Explicable (comme **SHAP**) : expliquer une prédiction individuelle en mesurant ce que chaque critère ajoute ou retire en dollars par an par rapport à une référence moyenne.

In [ ]:
#@title Simulateur de profil ANES 2024 (modifiez les curseurs puis cliquez sur ▶)
education = 14 #@param {type:"slider", min:6, max:20, step:1}
age = 52 #@param {type:"slider", min:18, max:80, step:1}
employed_label = "1 (En emploi)" #@param ["0 (Hors emploi)", "1 (En emploi)"]
married_label = "1 (Marié·e)" #@param ["0 (Non marié·e)", "1 (Marié·e)"]
female_label = "0 (Homme)" #@param ["0 (Homme)", "1 (Femme)"]
minority_label = "0 (Blanc non-hisp.)" #@param ["0 (Blanc non-hisp.)", "1 (Minorité racisée)"]

employed = int(employed_label[0])
married = int(married_label[0])
female = int(female_label[0])
minority = int(minority_label[0])

my_profile = pd.DataFrame(
    [
        {
            "education": education,
            "age": age,
            "employed": employed,
            "married": married,
            "female": female,
            "minority": minority,
        }
    ]
)

raw_pred = float(model_full.predict(my_profile).iloc[0])
pred = float(np.clip(raw_pred, 2500.0, 275000.0))
baseline_income = float(df["annual_income"].mean())

feature_names = ["education", "age", "employed", "married", "female", "minority"]
short_labels = {
    "education": "Études (education)",
    "age": "Âge (age)",
    "employed": "En emploi (employed)",
    "married": "Marié·e (married)",
    "female": "Femme (female)",
    "minority": "Minorité (minority)",
}

contributions = []
print("=== PROFIL CRÉÉ ===")
print(
    f"Études = {education} ans | Âge = {age} ans | "
    f"Activité = {'En emploi (1)' if employed == 1 else 'Hors emploi (0)'} | "
    f"Statut = {'Marié·e (1)' if married == 1 else 'Non marié·e (0)'} | "
    f"Sexe = {'Femme (1)' if female == 1 else 'Homme (0)'} | "
    f"Origine = {'Minorité (1)' if minority == 1 else 'Blanc non-hisp. (0)'}"
)
print(f"\nRéférence moyenne de l'échantillon ANES 2024 : {baseline_income:,.0f} $ / an".replace(",", " "))
print("\nDécomposition par variable (par rapport au profil moyen) :")

for feat in feature_names:
    coef = float(model_full.params[feat])
    mean_val = float(df[feat].mean())
    val = float(my_profile[feat].iloc[0])
    contrib = coef * (val - mean_val)
    contributions.append((short_labels[feat], contrib, val, mean_val))
    direction = "hausse" if contrib >= 0 else "baisse"
    contrib_str = f"{contrib:+,.0f}".replace(",", " ")
    print(
        f"  - {short_labels[feat]:<30} : valeur = {val:>5.2f} (moy. {mean_val:>5.2f}) -> "
        f"{contrib_str:>9} $ / an ({direction})"
    )

print(f"\nRevenu annuel prédit (borné 2 500 à 275 000 $ / an) : {pred:,.0f} $ / an".replace(",", " "))
if raw_pred != pred:
    print(f"(Note : prédiction brute avant plafonnement = {raw_pred:,.0f} $ / an)".replace(",", " "))

# Graphique en barres divergentes des contributions (#dd1019 rouge = - $/an, #0072bc bleu = + $/an)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plot_names = [item[0] for item in contributions][::-1]
plot_vals = [item[1] for item in contributions][::-1]
plot_colors = ["#0072bc" if v >= 0 else "#dd1019" for v in plot_vals]

fig, ax = plt.subplots(figsize=(9.0, 4.5))
bars = ax.barh(plot_names, plot_vals, color=plot_colors, height=0.58)
ax.axvline(0.0, color="#333333", linewidth=1.2)

max_abs = max([abs(v) for v in plot_vals] + [15000.0])
ax.set_xlim(-max_abs * 1.48, max_abs * 1.48)

for bar, val in zip(bars, plot_vals):
    offset = max_abs * 0.03 if val >= 0 else -max_abs * 0.03
    ha = "left" if val >= 0 else "right"
    label_val = f"{val:+,.0f} $ / an".replace(",", " ")
    ax.text(
        val + offset,
        bar.get_y() + bar.get_height() / 2,
        label_val,
        va="center",
        ha=ha,
        fontsize=10,
        fontweight="bold",
    )

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{int(x):+,} $".replace(",", " ")))
ax.set_title(
    f"Explication locale : revenu prédit = {pred:,.0f} $ / an (moyenne = {baseline_income:,.0f} $ / an)".replace(",", " "),
    fontsize=12.5,
    fontweight="bold",
)
ax.set_xlabel("Contribution en $ / an par rapport au profil moyen (rouge = - $ / an, bleu = + $ / an)")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

### À vous de jouer : 2 défis Hack-Time (5 min)

1. **Défi 1 (Audit de biais, toutes choses égales par ailleurs) :** dans le formulaire ci-dessus, basculez d'abord `female_label` de `0 (Homme)` à `1 (Femme)` sans rien changer d'autre puis relancez la cellule (**▶**). Vérifiez que le revenu prédit baisse **exactement** du coefficient de l'Étape 3 (`-8 007 $ / an`, tant que la prédiction reste dans `[2 500 ; 275 000] $ / an`). Basculez ensuite `minority_label` de `0 (Blanc non-hisp.)` à `1 (Minorité racisée)` pour constater la baisse supplémentaire exacte (`-10 328 $ / an`).
2. **Défi 2 (Contrefactuel et boîte noire) :** partez d'un profil modeste (par exemple `education = 10`, `age = 35`, `employed = 0`, `married = 0`, `female = 1`, `minority = 1`) et cherchez quel levier minimal (`education`, `employed`, `married`) le rapproche de `100 000 $ / an`. Testez ensuite un profil à `30` ans face à `70` ans dans l'**Onglet 4 (« Deep Learning », `329` paramètres)** de l'application web pour observer la courbe non linéaire de l'âge face au `-227 $ / an` fixe de la régression !

## Étape 5 : De `7` paramètres à `70 milliards` : pourquoi l'XAI ?

Dans ce notebook, notre modèle à **7 paramètres** reste une boîte de verre : ses poids se lisent sur une petite table. Mais dès que l'on cherche à capturer des courbures et des interactions non linéaires dans les données réelles (ou à traiter du texte et des images), le nombre de paramètres explose et l'explicabilité change de nature.

In [ ]:
xai_scale = pd.DataFrame(
    [
        {
            "Modèle": "Régression simple (Étape 2)",
            "Paramètres": "2",
            "R² / RMSE": "0.18 / 70 289 $ / an",
            "Transparence": "Boîte de verre (100 % lisible)",
            "Comment on l'explique": "Lecture directe de la constante et de la pente",
        },
        {
            "Modèle": "Régression multiple (Étape 3)",
            "Paramètres": "7",
            "R² / RMSE": "0.31 / 64 267 $ / an",
            "Transparence": "Boîte de verre (100 % lisible)",
            "Comment on l'explique": "Coefficients toutes choses égales par ailleurs et contributions exactes en $ / an",
        },
        {
            "Modèle": "Réseau de neurones de l'app (MLP 6 -> 16 -> 12 -> 1)",
            "Paramètres": "329",
            "R² / RMSE": "0.34 / 62 837 $ / an",
            "Transparence": "Boîte noire (non linéaire)",
            "Comment on l'explique": "Sondes après coup (LIME, SHAP : perturber chaque variable localement)",
        },
        {
            "Modèle": "Grand modèle de langage (LLM)",
            "Paramètres": "70 Md à 1 000+ Md",
            "R² / RMSE": "Non comparable",
            "Transparence": "Boîte noire massive",
            "Comment on l'explique": "Inspection des circuits internes et sondes ciblées",
        },
    ]
)

xai_scale

### 3 repères méthodologiques à retenir

1. **Boîte de verre vs explication *post-hoc* :** à `2` ou `7` paramètres, la formule *est* l'explication. Dès qu'un modèle empile des couches non linéaires (`329` paramètres dans le MLP de l'application, et plus encore dans un LLM à `70 milliards+`), on ne peut plus lire les poids directement : l'XAI doit construire une approximation locale *après coup* (*post-hoc*).
2. **Biais historiques, variables proxy et AI Act européen :** l'audit transparent révèle immédiatement `-8 007 $ / an` sur `female` et `-10 328 $ / an` sur `minority` (`p < 0.001`), mais supprimer ces colonnes d'une boîte noire ne suffit pas : des **variables proxy** (`education`, `married`, `employed`) continuent de transporter les inégalités historiques (comme dans le cas COMPAS). C'est pourquoi l'**AI Act européen** impose audit de biais, traçabilité et contrôle humain sur les systèmes à haut risque (recrutement, crédit, protection sociale).
3. **Réalisme du `R² = 0.31` sur de vraies données humaines et non-causalité :** sur une enquête réelle comme `ANES 2024`, expliquer `31.3 %` de la variance des revenus du foyer avec `6` variables est réaliste, car de nombreux facteurs individuels ne sont pas mesurés (métier exact, patrimoine, heures travaillées, région, second revenu du conjoint). De plus, une corrélation observée (`+10 998 $ / an` par année d'études ou `+48 093 $ / an` pour `married`) ne prouve jamais à elle seule un mécanisme causal isolé.

---

### Note sur l'usage de l'IA (AI Disclaimer)

Certaines parties de ce notebook (mise en forme du code pédagogique, textes explicatifs et visualisations) ont été conçues avec l'assistance d'outils d'IA générative. L'ensemble des analyses statistiques sur `ANES 2024`, des équations et des interprétations a été vérifié et validé sous la responsabilité éditoriale et scientifique de l'intervenant.